In [ ]:
import os
import torch
import torchvision
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# ── Paths ──────────────────────────────────────────────────────────────────
# Switch this depending on environment
KAGGLE = False

if KAGGLE:
    DATA_ROOT = "/kaggle/input/ff-c23"
else:
    DATA_ROOT = r"C:\\Users\\fmatt\\OneDrive\\Desktop\\AI_lab\\deepfake-detection\\data"

# ── Device ─────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Data root: {DATA_ROOT}")

Using device: cpu
Data root: C:\\Users\\fmatt\\OneDrive\\Desktop\\AI_lab\\deepfake-detection\\data


In [6]:
import numpy as np
import PIL
from itertools import product

# define data paths
splits = ["train", "test", "val"]
classes= ["real", "fake"]
paths = [os.path.join(DATA_ROOT, split, cls) for split, cls in product(splits, classes)]

# filling folders with 5 dummy images
for path in paths:
    os.makedirs(path, exist_ok=True)
    for i in range(5):
        np_arr = np.random.randint(0,255, (224,224,3), dtype=np.uint8)
        img = PIL.Image.fromarray(np_arr)
        img.save(os.path.join(path, f"dummy_{i}.jpg"))

In [ ]:
# transformations for train/val/test

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform  = transforms.Compose([transforms.RandomResizedCrop(224),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean, std)])

val_transform = transforms.Compose([transforms.Resize(256),
                                   transforms.CenterCrop(224),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean, std)])

test_transform = transforms.Compose([transforms.Resize(256),
                                   transforms.CenterCrop(224),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean, std)])

In [8]:
train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")
test_path = os.path.join(DATA_ROOT, "test")

train_dataset = ImageFolder(train_path, transform=train_transform)
val_dataset = ImageFolder(val_path, transform=val_transform)
test_dataset = ImageFolder(test_path, transform=test_transform)

train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False)


In [9]:
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

Train: 10 | Val: 10 | Test: 10
Classes: ['fake', 'real']


One important thing to note: ImageFolder assigned fake=0 and real=1 alphabetically. Keep this in mind later when interpreting model outputs.

In [11]:
X, y = next(iter(train_dataloader))
X.shape, y.shape

(torch.Size([10, 3, 224, 224]), torch.Size([10]))